In [ ]:
# from datetime import datetime, timezone

# import pandas as pd
# import pymongo
# from pythonbq import pythonbq
# from sqlalchemy import create_engine, text
# import numpy as np
# from access.keys import bigquery_project
# from access.utils import get_documentdb_uri


# mongo_uri = get_documentdb_uri(instance=3, port=9990)
# client = pymongo.MongoClient(mongo_uri)
# mydb = client["dumpdb"]
# mycol = mydb["PartnerAgentMappingDump"]

# USER = "datateam"
# PASSWORD = "datateam&10avable"
# HOST = "datateam-lovable.coe21b2cs9fz.ap-south-1.rds.amazonaws.com"
# PORT = 3306
# DB_NAME = "vyapar_lovable"

# engine = create_engine(
#     f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}",
#     pool_pre_ping=True,
#     pool_recycle=3600
# )

# try:
#     with engine.connect() as conn:
#         result = conn.execute(text("SELECT 1"))
#         print("Connection successful:", result.fetchall())
# except Exception as e:
#     print("Connection failed:", str(e))


# myProject = pythonbq(
#     bq_key_path=bigquery_project['key_path'],
#     project_id=bigquery_project['id']
# )


TP_AGENT_DETAILS_QUERY = """
SELECT
    t1.agent_id,
    t1.agent_user_id,
    t1.agent_name,
    t1.cluster_id,
    t2.cluster_name,
    t2.sub_team_1_id,
    t2.sub_team_2_id,
    3 AS designationPriority,
    t1.team_lead_id,
    t1.team_lead_name,
    t1.created_at,
    t1.updated_at
FROM ipt_utility_mappings t1
LEFT JOIN ipt_utility_clusters t2
    ON t1.cluster_id = t2.id
WHERE
    t1.is_active = 1
    AND t1.designation_name = "Activation Agent"  and sub_team_1_id =1
"""

TP_ACTIVATION_TO_HUNTING_QUERY = """
SELECT *
FROM (
    SELECT *
    FROM (

WITH partner_details AS (
    SELECT
        agent_email,
        team_lead,
        assigned_date,
        d_id,
        dis_created_at AS onboarded_dt,
        UPPER(TRIM(referral_code)) AS referral_code,statename
    FROM `vyapar_mysql.vyapar_partners_details`
    where agent_email not in ('roopdiya@vyapar.roopdiya@vyapar.com', 'manasa.m@vyapar.com', 'sharmilap@vyapar.com', 'chikagoudru.shashank@vyapar.com', 'vaishnavi.patrikar@vyapar.com', 'nishad.abdulrehman@vyapar.com', 'ashish.kumar@vyapar.com', 'syedisrar@vyapar.com')
    and lower(cluster_name) like 'ca%'
),

lic_details AS (
    SELECT
        DATE_TRUNC(DATE(payment_success_date), MONTH) AS payment_month,
        DATE(payment_success_date) AS payment_date,
        lic_id,
        platform,
        UPPER(TRIM(referral_code)) AS referral_code
    FROM `vyapar_mysql.vyapar_detailed_licenses`
    WHERE platform = 2
      AND workflow IN (1, 6)
),

activation_details AS (
    SELECT
        p.*,
        MIN(l.payment_date) AS activation_dt,Count(distinct lic_id) lic_cnt
    FROM partner_details p
    LEFT JOIN lic_details l
        ON p.referral_code = l.referral_code
       AND p.onboarded_dt <= l.payment_date
    GROUP BY ALL
    -- HAVING lic_cnt<3
)


SELECT * FROM activation_details
WHERE cast(d_id as string) not in (SELECT DISTINCT distributorid FROM `vyapar_data.data_hunting_logs`)
and activation_dt is null)
)
WHERE
    activation_dt IS NULL
    AND DATE_DIFF(CURRENT_DATE(), onboarded_dt, day) = 60
"""

M0_ACTIVATION_INACTIVE_HUNTING_QUERY = """
SELECT
    d_id AS distributorId,
    2 AS subTeam1Id,
    1 AS subTeam2Id,
    statename,
    3 as designationPriority,
FROM (
    SELECT DISTINCT pd.*
    FROM (
        SELECT distinct d_id, statename, referral_code, dis_created_at
        FROM `vyapar_mysql.vyapar_partners_details`
        WHERE cluster_name = 'M0 Activation'
            AND ipt_team = 'hunting'
    ) AS pd
    LEFT JOIN (
        SELECT referral_code, payment_success_date
        FROM `vyapar_mysql.vyapar_detailed_licenses`
        WHERE workflow IN (1, 6)
            AND platform = 2
    ) AS dl
        ON UPPER(TRIM(pd.referral_code)) = UPPER(TRIM(dl.referral_code))
        AND DATE(pd.dis_created_at) <= DATE(dl.payment_success_date)
    WHERE payment_success_date IS NULL
)
WHERE DATE_DIFF(CURRENT_DATE(), DATE(dis_created_at), DAY) > 90
"""

M0_ACTIVATION_ACTIVE_HUNTING_QUERY = """
SELECT
    d_id AS distributorId,
    2 AS subTeam1Id,
    1 AS subTeam2Id,
    statename,
    3 AS designationPriority
FROM (
    SELECT DISTINCT pd.*
    FROM (
        SELECT d_id, referral_code, dis_created_at, statename
        FROM `vyapar_mysql.vyapar_partners_details`
        WHERE cluster_name = 'M0 Activation'
            AND ipt_team = 'hunting'
    ) AS pd
    LEFT JOIN (
        SELECT referral_code, payment_success_date
        FROM `vyapar_mysql.vyapar_detailed_licenses`
        WHERE workflow IN (1, 6)
            AND platform = 2
    ) AS dl
        ON UPPER(TRIM(pd.referral_code)) = UPPER(TRIM(dl.referral_code))
        AND DATE(pd.dis_created_at) <= DATE(dl.payment_success_date)
    WHERE payment_success_date IS NOT NULL
        AND DATE(payment_success_date) < CURRENT_DATE()
)
"""

REVIVAL_ACTIVITY_HUNTING_QUERY = """
SELECT
    a.d_id AS distributorId,
    2 AS subTeam1Id,
    CASE WHEN r.lead_type = 'hunting' THEN 1 ELSE 2 END AS subTeam2Id,
    a.statename,
    CASE WHEN r.lead_type = 'farming' THEN 4 ELSE 3 END AS designationPriority
FROM (
    SELECT DISTINCT a.*
    FROM (
        SELECT distinct  d_id, d_phone, referral_code, dis_created_at, statename
        FROM `vyapar_mysql.vyapar_partners_details`
        WHERE agent_email = 'revival@vyapar.com'
    ) AS a
    LEFT JOIN (
        SELECT DISTINCT
            UPPER(TRIM(ref_code)) AS ref_code
        FROM `vyapar_mysql.vyapar_data_ipt_main`
        WHERE COALESCE(
            SAFE_CAST(assigned_date AS DATE),
            SAFE.PARSE_DATE('%d/%m/%Y', CAST(assigned_date AS STRING))
        ) = CURRENT_DATE() - interval 1 day
    ) AS ip
        ON UPPER(TRIM(a.referral_code)) = ip.ref_code
    LEFT JOIN (
        SELECT DISTINCT
            UPPER(TRIM(referral_code)) AS ref_code
        FROM `vyapar_new.partner_facing_login`
        WHERE COALESCE(
            SAFE_CAST(login_date AS DATE),
            SAFE.PARSE_DATE('%d/%m/%Y', CAST(login_date AS STRING))
        ) = CURRENT_DATE() - interval 1 day
    ) AS fc
        ON UPPER(TRIM(a.referral_code)) = fc.ref_code
    LEFT JOIN (
        SELECT DISTINCT source_id
        FROM (
            SELECT
                * 
            FROM (
                SELECT
                    source_id, __is_deleted,source_type, callDirection, starttime,
                    ROW_NUMBER() OVER (
                        PARTITION BY _id
                        ORDER BY __hevo__ingested_at DESC
                    ) AS rnk
                FROM `vyapar-faf62.mongo_dumpdb.vyapar_calls`
                WHERE DATE_TRUNC(DATE(_v_partitionDate), MONTH)
                    >= DATE_TRUNC(CURRENT_DATE() - INTERVAL 1 DAY, MONTH)
            )
            WHERE (__is_deleted IS FALSE OR __is_deleted IS NULL)
                AND rnk = 1
        )
        WHERE source_type IN (
            'iptpo_alternate',
            'ipt0_po',
            'iptpo_tp_alternate',
            'iptpo'
        )
            AND callDirection = 'inbound'
            AND COALESCE(
                SAFE_CAST(starttime AS DATE),
                SAFE.PARSE_DATE(
                    '%d/%m/%Y',
                    SUBSTR(CAST(starttime AS STRING), 1, 10)
                )
            ) = CURRENT_DATE() - interval 1 day
    ) AS cal
        ON SAFE_CAST(cal.source_id AS INT64) = a.d_id
    LEFT JOIN (
        SELECT DISTINCT phone_number
        FROM `vyapar_data.ad_realtime_activity`
        WHERE event IN ('support_incoming')
            AND COALESCE(
                SAFE_CAST(event_date AS DATE),
                SAFE.PARSE_DATE(
                    '%d/%m/%Y',
                    SUBSTR(CAST(event_date AS STRING), 1, 10)
                )
            ) = CURRENT_DATE() - interval 1 day
    ) AS sup
        ON RIGHT(a.d_phone, 10) = RIGHT(sup.phone_number, 10)
    WHERE ip.ref_code IS NOT NULL
        OR fc.ref_code IS NOT NULL
        OR cal.source_id IS NOT NULL
        OR sup.phone_number IS NOT NULL
) AS a
LEFT JOIN (
    SELECT *
    FROM vyapar_data.vyapar_ipt_revival_email_account
) AS r
    ON a.d_id = r.d_id
WHERE r.lead_type IN ('farming', 'hunting')
"""

MAPPING_COLUMNS = [
    'distributorId',
    'subTeam1Id',
    'subTeam2Id',
    'stateId',
    'designationPriority',
    'isProcessed',
    'createdAt'
]


def get_activation_to_hunting_mapping():
    agent_details_df = pd.read_sql(TP_AGENT_DETAILS_QUERY, engine)
    lead_flow_df = myProject.query(sql=TP_ACTIVATION_TO_HUNTING_QUERY)
    lead_flow_df = pd.DataFrame(lead_flow_df)

    if len(lead_flow_df) == 0:
        return pd.DataFrame(columns=MAPPING_COLUMNS)

    final_df = lead_flow_df.merge(
        agent_details_df[
            [
                'agent_id',
                'cluster_name',
                'sub_team_1_id',
                'sub_team_2_id',
                'designationPriority'
            ]
        ],
        how='inner',
        left_on='agent_email',
        right_on='agent_id'
    )

    if len(final_df) == 0:
        return pd.DataFrame(columns=MAPPING_COLUMNS)

    final_df['createdAt'] = datetime.now(timezone.utc)
    final_df['isProcessed'] = False
    final_df['d_id'] = final_df['d_id'].astype(str)

    mapping_df = final_df[
        [
            'd_id',
            'sub_team_1_id',
            'sub_team_2_id',
            'statename',
            'designationPriority',
            'isProcessed',
            'createdAt'
        ]
    ].rename(
        columns={
            'd_id': 'distributorId',
            'sub_team_1_id': 'subTeam1Id',
            'sub_team_2_id': 'subTeam2Id',
            'statename': 'stateId'
        }
    )

    return mapping_df


# def get_mapping_from_bq_query(query):
#     mapping_df = myProject.query(sql=query)
#     mapping_df = pd.DataFrame(mapping_df)

#     if len(mapping_df) == 0:
#         return pd.DataFrame(columns=MAPPING_COLUMNS)

#     mapping_df['distributorId'] = mapping_df['distributorId'].astype(str)
#     if 'designationPriority' not in mapping_df.columns:
#         print("designationPriority not found in query result. Skipping data push.")
#         return pd.DataFrame(columns=MAPPING_COLUMNS)
#     mapping_df['isProcessed'] = False
#     mapping_df['createdAt'] = datetime.now(timezone.utc)
#     mapping_df = mapping_df.rename(columns={'statename': 'stateId'})

#     return mapping_df[MAPPING_COLUMNS]


# def push_mapping_to_mongo(mapping_df, query_name):
#     if len(mapping_df) == 0:
#         print("NO", query_name, "data to push")
#         return

#     mapping_df = mapping_df.copy()
#     mapping_df['sourceQuery'] = query_name
#     mycol.insert_many(mapping_df.to_dict(orient='records'))
#     print(len(mapping_df), query_name, "records pushed to mongo successfully")


# push_mapping_to_mongo(
#     get_activation_to_hunting_mapping(),
#     "activation_to_hunting_60_days"
# )

# push_mapping_to_mongo(
    get_mapping_from_bq_query(M0_ACTIVATION_INACTIVE_HUNTING_QUERY),
    "m0_activation_inactive_hunting"
)

push_mapping_to_mongo(
    get_mapping_from_bq_query(REVIVAL_ACTIVITY_HUNTING_QUERY),
    "revival_activity_hunting"
)

if datetime.now().day == 1:
    push_mapping_to_mongo(
        get_mapping_from_bq_query(M0_ACTIVATION_ACTIVE_HUNTING_QUERY),
        "m0_activation_active_hunting"
    )
else:
    print("m0_activation_active_hunting leads will be pushed on 1st not today")

print("========================DONE================")